In [3]:
## BoundedHybridGenerator developed by Bisma Naeem as part of MammoWave breast cancer research work.
class BoundedHybridGenerator:
    ## constructor function for the class BoundedHybridGenerator
    def __init__(self, real_df , feature_cols):
        ## storing the names of feature columns
        self.feature_cols=feature_cols
        self.real=real_df
        ## by keeping only cancer samples
        self.real_cancer=real_df[real_df["cancer"]==1][feature_cols].values
        ## basic stats from real cancer data
        self.real_min=self.real_cancer.min(axis=0)
        self.real_max=self.real_cancer.max(axis=0)
        self.real_mean=self.real_cancer.mean(axis=0)
        self.real_std=self.real_cancer.std(axis=0)
        ## setting soft boundaries (to avoid the extreme values)
        self.lb=self.real_mean-3*self.real_std
        self.ub=self.real_mean+3*self.real_std
        ## nearest neighbors model (for interpolation)
        self.nn=NearestNeighbors(n_neighbors=5).fit(self.real_cancer)
    ## interpolate sample generation
    def generate_interpolated(self, n):
        ## empty list for the samples
        samples=[]
        for _ in range(n):
            ## finding 5 nearest neighbors to patient i
            i=np.random.randint(len(self.real_cancer))
            neighbors=self.nn.kneighbors([self.real_cancer[i]],return_distance=False)[0]
            ## pick one of the 4 remaining neighbors
            j=np.random.choice(neighbors[1:])
            ## pick a random mixing weight between 0.2 and 0.8 to avoid extremes
            lam=np.random.uniform(0.2,0.8)
            samples.append(lam*self.real_cancer[i]+(1-lam)*self.real_cancer[j])
            ## convert collected samples into a numpy array
        return np.array( samples )
    ## to produce synthetic cancer samples by mixing 2 sources
    def generate_hybrid(self, n, vae_samples, blend):
        ## how many exact samples to take from the vae pool
        n_vae=int(n*blend)
        ## all the remaining samples come from interpolation
        n_int=n-n_vae
        ## to randomly draw n_vae samples from vae pool without replacement (no duplicates)
        vae_part=vae_samples[np.random.choice(len(vae_samples),n_vae , replace=False)]
        ## generate the interpolated part
        int_part=self.generate_interpolated(n_int)
        #stack both sources vertically into one combined
        X=np.vstack([vae_part,int_part])
        for i in range(X.shape[1]):
            ## clamping to soft statistical bounds
            X[:,i]=np.clip(X[:,i],self.lb[i],self.ub[i])
            ## clamping to real observed min/max
            X[:,i]=np.clip(X[:,i],self.real_min[i],self.real_max[i])
        return X
## Boundary aware filtering
def boundary_filter(real_df, synth_df, feature_cols, margin= 0.2):
    Xr=real_df[feature_cols].values
    yr=real_df["cancer"].values
    scaler=StandardScaler()
    Xr=scaler.fit_transform(Xr)
    svm=SVC(kernel="rbf",C=1.0,gamma="scale",class_weight="balanced",probability=True,random_state=RANDOM_STATE,)
    svm.fit(Xr,yr)
    Xs=scaler.transform(synth_df[feature_cols].values)
    probs=svm.predict_proba(Xs)[:,1]
    keep=(probs>0.5-margin) & (probs < 0.5 + margin)
    return synth_df[keep]
        